# 🎬 Animé → TikTok en 4K / 120-240 fps

Ce notebook tourne sur **Kaggle** (GPU T4/P100) ou **Google Colab** (GPU T4 + Google Drive).

1. **Analyse** des épisodes : plans, son, mouvement ; génériques et récaps ignorés automatiquement
2. **Sélection** des moments forts, que vous validez
3. **Rendu** des extraits : vertical 9:16, interpolation RIFE (120/240 fps), agrandissement Real-ESRGAN (4K)
4. **Montage** selon le style du jour (`brut`, `hype`, `cinematique`, ou vos propres styles dans `config.yaml`)

**Avant de lancer :**
- Kaggle : *Settings → Accelerator → GPU* et *Internet → On*. Ajoutez vos épisodes comme *Dataset* (*Add Input*).
- Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*. Déposez les épisodes dans `Mon Drive/anime_tiktok/episodes`.

In [ ]:
import os, sys, glob

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/amola07/mon_dashboard_client.git'   # dépôt du pipeline

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/anime_tiktok'   # tout ce qui doit survivre à la session va sur Drive
    CODE = '/content/pipeline'
    TMP = '/content/tmp'                            # images temporaires : disque local (rapide)
    EPISODES = f'{BASE}/episodes'
elif ON_KAGGLE:
    BASE = '/kaggle/working'
    CODE = '/kaggle/working/pipeline'
    TMP = '/kaggle/tmp'
    # Les épisodes de vos Datasets (/kaggle/input) sont liés dans un dossier modifiable,
    # pour pouvoir en ajouter depuis l'application.
    EPISODES = f'{BASE}/episodes'
    os.makedirs(EPISODES, exist_ok=True)
    VIDEO_EXTS = ('.mkv', '.mp4', '.webm', '.mov', '.avi', '.m4v', '.ts')
    for src in glob.glob('/kaggle/input/**/*', recursive=True):
        dst = os.path.join(EPISODES, os.path.basename(src))
        if src.lower().endswith(VIDEO_EXTS) and not os.path.exists(dst):
            os.symlink(src, dst)
else:
    BASE = os.path.abspath('.')
    CODE = BASE
    TMP = f'{BASE}/work/tmp'
    EPISODES = f'{BASE}/input/episodes'

MUSIC = f'{BASE}/music'
WORK = f'{BASE}/work'
OUTPUT = f'{BASE}/output'
for d in (MUSIC, WORK, OUTPUT, TMP):
    os.makedirs(d, exist_ok=True)
print('Épisodes :', EPISODES)

In [ ]:
# Récupère (ou met à jour) le code du pipeline.
# Si le dépôt est privé, utilisez https://<token>@github.com/... ou rendez-le public.
if CODE != BASE:
    if os.path.exists(f'{CODE}/.git'):
        !git -C {CODE} pull -q
    else:
        !git clone -q {REPO_URL} {CODE}
# Web app (version iPhone) compilée par GitHub Actions : servie par la télécommande.
!rm -rf {CODE}/webapp && git clone -q --depth 1 -b web-build {REPO_URL} {CODE}/webapp || echo "Web app indisponible (branche web-build absente)"
!pip -q install -r {CODE}/requirements.txt
!bash {CODE}/scripts/setup_tools.sh
os.environ['PATH'] = f'{CODE}/tools/ffmpeg/bin:' + os.environ['PATH']
!nvidia-smi --query-gpu=name,memory.total --format=csv

## ⚙️ Réglages

Changez ici le style du jour, le fps et le nom de l'animé. Toute autre valeur de `config.yaml`
peut être surchargée dans `EXTRA` (ex. `'selection.clips_total=30'`).

In [ ]:
ANIME = 'Mon Anime'
STYLE = 'brut'          # brut | hype | cinematique
FPS = 120               # 120 ou 240
WIDTH, HEIGHT = 2160, 3840   # 4K vertical
EXTRA = [
    # 'selection.clips_total=30',
    # 'selection.max_seconds=30',
    # 'analysis.skip_start=5',
]

settings = [
    f'paths.episodes={EPISODES}', f'paths.music={MUSIC}', f'paths.work={WORK}',
    f'paths.output={OUTPUT}', f'paths.tmp={TMP}', f'paths.tools={CODE}/tools',
    f'render.fps={FPS}', f'render.width={WIDTH}', f'render.height={HEIGHT}',
    f'delivery.anime={ANIME!r}', *EXTRA,
]
SETS = ' '.join(f'--set "{s}"' for s in settings)
PIPE = f'cd {CODE} && python -m anime_tiktok'
!{PIPE} backends {SETS}

## 📱 Télécommande (application Anime TikTok Studio)

Avec `TELECOMMANDE = True`, cette cellule démarre le serveur de contrôle et **tout se pilote depuis le
téléphone** : analyse, validation des extraits, style du jour, rendu, visionnage et partage vers TikTok.
Les cellules suivantes (mode manuel) sont alors ignorées.

- Choisissez un **mot de passe** et un **sujet ntfy** (un nom unique, difficile à deviner), et entrez les mêmes
  dans l'application : elle retrouvera toute seule l'adresse du serveur, même quand elle change à chaque session.
- Kaggle : *Save Version → Save & Run All* fait tourner le notebook en arrière-plan jusqu'à 12 h,
  navigateur fermé.
- Colab : laissez l'onglet ouvert.
- Optionnel : abonnez-vous au même sujet dans l'application **ntfy** pour recevoir une notification à la
  fin de chaque rendu.
- iPhone : la web app est aussi servie par ce serveur, à l'adresse publique affichée ci-dessous
  (ou, adresse fixe, sur votre hébergement Cloudflare Pages / Netlify de la branche `web-build`).

In [ ]:
TELECOMMANDE = True
MOT_DE_PASSE = 'changez-moi'          # le même dans l'application
SUJET_NTFY = ''                       # ex. 'anime-tiktok-k7x2p9' : le même dans l'application

if TELECOMMANDE:
    assert MOT_DE_PASSE != 'changez-moi', 'Choisissez un mot de passe'
    assert SUJET_NTFY, 'Choisissez un sujet ntfy (nom unique)'
    !{PIPE} serve {SETS} --tunnel --token "{MOT_DE_PASSE}" --ntfy-topic "{SUJET_NTFY}"

## 1. Analyse des épisodes (mode manuel)
Quelques minutes pour 12 épisodes. Le résultat est gardé : relancer ne refait que les nouveaux épisodes.

In [ ]:
if not TELECOMMANDE:
    !{PIPE} analyze {SETS}

## 2. Sélection des moments forts

In [ ]:
if not TELECOMMANDE:
    !{PIPE} select {SETS}

import base64, csv
from IPython.display import HTML, display

def apercu():
    rows = list(csv.DictReader(open(f'{WORK}/selection/clips.csv', encoding='utf-8')))
    cards = []
    for r in rows:
        thumb = f"{WORK}/selection/thumbs/{r['id']}.jpg"
        img = base64.b64encode(open(thumb, 'rb').read()).decode() if os.path.exists(thumb) else ''
        style = '' if r['keep'] == '1' else 'opacity:.35'
        cards.append(f'''<div style="width:240px;{style}"><img src="data:image/jpeg;base64,{img}" style="width:100%">
<div style="font:12px sans-serif"><b>{r['id']}</b><br>ép. {r['episode']} · {float(r['start']):.0f}s → {float(r['end']):.0f}s
· score {float(r['score']):.2f}</div></div>''')
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px">' + ''.join(cards) + '</div>'))

if not TELECOMMANDE:
    apercu()

### Validation
Indiquez les extraits à **écarter** (leurs ids ci-dessus), puis exécutez la cellule.

In [ ]:
if not TELECOMMANDE:
    ECARTER = []   # ex. ['c03_ep02_00731', 'c11_ep07_00102']

    path = f'{WORK}/selection/clips.csv'
    rows = list(csv.DictReader(open(path, encoding='utf-8')))
    for r in rows:
        r['keep'] = '0' if r['id'] in ECARTER else '1'
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    print(sum(r['keep'] == '1' for r in rows), 'extraits retenus')
    apercu()

## 3 + 4. Rendu 4K / haute fréquence et montage
Les extraits déjà rendus sont réutilisés (changer de style ne refait que le montage, sauf si le recadrage change).
Pour le style `hype`, déposez une ou plusieurs musiques dans le dossier `music`.

In [ ]:
if not TELECOMMANDE:
    !{PIPE} make --style {STYLE} {SETS}

## 5. Livraison

In [ ]:
if not TELECOMMANDE:
    dernier = sorted(glob.glob(f'{OUTPUT}/*'))[-1]
    print('Vidéos prêtes :', dernier)
    for f in sorted(glob.glob(f'{dernier}/*.mp4')):
        print(' -', os.path.basename(f), f'{os.path.getsize(f) / 1e6:.0f} Mo')
    if ON_KAGGLE:
        import shutil
        shutil.make_archive(f'{BASE}/livraison', 'zip', dernier)
        print('Archive à télécharger : /kaggle/working/livraison.zip (onglet Output)')
    elif ON_COLAB:
        print('Les vidéos sont dans votre Google Drive : anime_tiktok/output')